In [1]:
import numpy as np

# Basic Operations


## Vectors

In [13]:
# create a 1D array
a_1d = np.array([1, 2, 3, 4])
b_1d = np.array([5, 6, 7, 8])
print("sum:", a_1d + b_1d)  # element-wise addition
print("product:", a_1d * b_1d)
print("shape:", a_1d.shape) 
# turn into row vectors
a_row = a_1d.reshape(1, -1)
b_row = b_1d.reshape(1, -1)
print("row vector shape:", a_row.shape)
print("row vector sum:", a_row + b_row)
print("row vector product:", a_row * b_row)

# turn into column vectors
a_col = a_1d.reshape(-1, 1)
b_col = b_1d.reshape(-1, 1)
print("column vector shape:", a_col.shape)
print("column vector sum:", a_col + b_col)

sum: [ 6  8 10 12]
product: [ 5 12 21 32]
shape: (4,)
row vector shape: (1, 4)
row vector sum: [[ 6  8 10 12]]
row vector product: [[ 5 12 21 32]]
column vector shape: (4, 1)
column vector sum: [[ 6]
 [ 8]
 [10]
 [12]]


## Matrix

numpy can create matrix in several ways

In [14]:
# specify the matrix
matrix = np.array([a_1d, b_1d])
print("matrix shape:", matrix.shape)

# outer product of two vectors
matrix_outer_product = np.outer(a_col, b_col)
print("outer product shape:", matrix_outer_product.shape)
print("outer product:\n", matrix_outer_product)

# matrix multiplication
matrix_mult = a_col @ b_row
print("matrix multiplication shape:", matrix_mult.shape)
print("matrix multiplication:\n", matrix_mult)


matrix shape: (2, 4)
outer product shape: (4, 4)
outer product:
 [[ 5  6  7  8]
 [10 12 14 16]
 [15 18 21 24]
 [20 24 28 32]]
matrix multiplication shape: (4, 4)
matrix multiplication:
 [[ 5  6  7  8]
 [10 12 14 16]
 [15 18 21 24]
 [20 24 28 32]]


## BroadCasting

When operating on two arrays, Numpy compares their shapes element-wise. It starts with the trailing (i.e. rightmost) dimension and works its way left. Two dimension are compatible when   
1. they are equal
2. one of them is 1
If these conditions are not met, a ```ValueError: operands could not be broadcast together``` exception is thrown.

In [17]:
# consider a batch of images (B x W x H x C)
batch_size = 10
width = 28
height = 28
channels = 3
images = np.random.rand(batch_size, width, height, channels)
print("images shape:", images.shape)

# add another single image to them
single_image = np.random.rand(width, height, channels)
print("single image shape:", single_image.shape)

# broadcasting will automatically expand single_image to (B x W x H x C)
images_plus_single = images + single_image
print("resulting shape after addition:", images_plus_single.shape)

# failed case: incompatible shapes
# incompatible_image = np.random.rand(2, width, height, channels)  # missing batch dimension
# images_plus_incompatible = images + incompatible_image  # This will raise an error

images shape: (10, 28, 28, 3)
single image shape: (28, 28, 3)
resulting shape after addition: (10, 28, 28, 3)


# Advanced Usage

For Basic usage, just ask LLM or look at the dictionary, here we cover more interesting usages.

## Is vectorize good enough?

For each row $x$ of a matrix $X \in \mathbb{R}^{N \times D}$
1. Compute mean $\mu$
2. Compute standard devviation $\sigma$
3. Normalize $(x - \mu) / \sigma$
4. Count how many normalized values exceed a threshold $\tau$

In [23]:
def numpy_version_loop(X, thresh=1.0):
    N, D = X.shape
    out = np.zeros(N, dtype=np.int32)

    for i in range(N):
        # mean
        mu = 0.0
        for j in range(D):
            mu += X[i, j]
        mu /= D

        # std
        var = 0.0
        for j in range(D):
            diff = X[i, j] - mu
            var += diff * diff
        std = np.sqrt(var / D) + 1e-12

        # normalize + count
        cnt = 0
        for j in range(D):
            z = (X[i, j] - mu) / std
            if z > thresh:
                cnt += 1

        out[i] = cnt

    return out

In [18]:


def numpy_version(X, thresh=1.0):
    mu = X.mean(axis=1, keepdims=True)          # pass 1
    std = X.std(axis=1, keepdims=True) + 1e-12  # pass 2
    Z = (X - mu) / std                          # pass 3 (alloc)
    return (Z > thresh).sum(axis=1)             # pass 4



In [19]:
from numba import njit
import numpy as np

@njit
def numba_version(X, thresh=1.0):
    N, D = X.shape
    out = np.zeros(N, dtype=np.int32)

    for i in range(N):
        # mean
        mu = 0.0
        for j in range(D):
            mu += X[i, j]
        mu /= D

        # std
        var = 0.0
        for j in range(D):
            diff = X[i, j] - mu
            var += diff * diff
        std = np.sqrt(var / D) + 1e-12

        # normalize + count
        cnt = 0
        for j in range(D):
            z = (X[i, j] - mu) / std
            if z > thresh:
                cnt += 1

        out[i] = cnt

    return out


In [24]:
import time

N, D = 20000, 512
X = np.random.randn(N, D).astype(np.float64)

t0 = time.perf_counter()
a = numpy_version(X)
t1 = time.perf_counter()


b = numba_version(X)
t2 = time.perf_counter()

c = numpy_version_loop(X)
t3 = time.perf_counter()



print(f"NumPy vectorized: {t1 - t0}")
print(f"Numba fused loop: {t2 - t1}")
print(f"NumPy loop version: {t3 - t2}")
print(f"max diff: {np.max(np.abs(a - b))}")

NumPy vectorized: 0.0529549999628216
Numba fused loop: 0.022490708041004837
NumPy loop version: 7.6891026250086725
max diff: 0
